Optimizers in Artificial Neural Networks

Data Science Masters 2.0 – Assignment

Part 1 – Understanding Optimizers
1. Role of optimization algorithms in neural networks

Optimization algorithms update the trainable parameters (weights and biases) of a neural network in order to minimize a loss function.

They are necessary because:

Neural networks contain thousands or millions of parameters

There is no closed-form solution for minimizing the loss

We must rely on iterative numerical methods

In short, an optimizer controls:

how fast the model learns

how stable the training process is

how well the final model converges

2. Gradient Descent and its variants

Gradient Descent (GD) updates parameters in the opposite direction of the gradient of the loss.

𝜃
=
𝜃
−
𝜂
∇
𝜃
𝐽
(
𝜃
)
θ=θ−η∇
θ
	​

J(θ)

where

𝜂
η = learning rate

Variants
Method	Update based on	Main idea
Batch Gradient Descent	Entire dataset	Accurate but slow
Stochastic Gradient Descent (SGD)	One sample	Fast but noisy
Mini-batch Gradient Descent	Small batch	Balance of speed and stability
Trade-offs
Aspect	Batch GD	SGD	Mini-batch
Convergence speed	Slow	Fast updates	Fast and stable
Memory	High	Very low	Moderate
Noise	Low	High	Medium
3. Challenges of traditional gradient descent

Main problems:

Slow convergence

Sensitive to learning rate

Oscillation in narrow valleys

Poor behavior on ill-conditioned problems

Can get stuck in flat regions or saddle points

Modern optimizers solve these by:

using momentum

using adaptive learning rates

keeping running statistics of gradients

4. Momentum and learning rate
Learning rate

Controls the step size of updates.

Too large → divergence

Too small → very slow learning

Momentum

Momentum accumulates past gradients to smooth updates.

𝑣
𝑡
=
𝛽
𝑣
𝑡
−
1
+
(
1
−
𝛽
)
𝑔
𝑡
v
t
	​

=βv
t−1
	​

+(1−β)g
t
	​


Effect:

reduces oscillations

accelerates convergence in consistent directions

Both directly affect:

convergence speed

stability

final model performance

Part 2 – Optimizer Techniques
1. Stochastic Gradient Descent (SGD)

SGD updates the parameters using one mini-batch at a time.

Advantages

Very memory efficient

Works well for large datasets

Helps escape shallow local minima due to noise

Limitations

Noisy updates

Sensitive to learning rate

Slower convergence near optimum

Best suited for

large-scale problems

online learning

when generalization is more important than fast convergence

2. Adam optimizer

Adam combines:

momentum (first moment)

adaptive learning rate (second moment)

It keeps:

exponential moving average of gradients

exponential moving average of squared gradients

Benefits

fast convergence

little tuning required

works well on sparse and noisy gradients

Drawbacks

sometimes worse generalization than SGD

may converge to sharper minima

3. RMSprop optimizer

RMSprop adapts the learning rate by maintaining a moving average of squared gradients.

Main idea:

𝜃
=
𝜃
−
𝜂
𝐸
[
𝑔
2
]
+
𝜖
𝑔
θ=θ−
E[g
2
]
	​

+ϵ
η
	​

g
How it helps

avoids aggressive updates

stabilizes training when gradients vary in magnitude

Comparison with Adam
Aspect	RMSprop	Adam
Momentum	No (only adaptive lr)	Yes
Speed	Fast	Very fast
Stability	High	High
General use	Good	Very popular default

Adam can be seen as RMSprop + momentum.

Part 3 – Applying Optimizers
Dataset and Model

We will use the Wine Quality dataset and build a simple neural network for classification.

1. Implementation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

Load and prepare data

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

data = pd.read_csv(url, sep=';')

X = data.drop("quality", axis=1)
y = data["quality"]

# Convert to classification problem
y = (y >= 6).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Model builder function

In [ ]:
def build_model(optimizer):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

Training with SGD

In [ ]:
sgd_model = build_model(
    tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
)

history_sgd = sgd_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

Training with Adam

In [ ]:
adam_model = build_model(
    tf.keras.optimizers.Adam(learning_rate=0.001)
)

history_adam = adam_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

Training with RMSprop

In [ ]:
rms_model = build_model(
    tf.keras.optimizers.RMSprop(learning_rate=0.001)
)

history_rms = rms_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

Evaluation

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    return accuracy_score(y_test, y_pred)

acc_sgd = evaluate(sgd_model, X_test, y_test)
acc_adam = evaluate(adam_model, X_test, y_test)
acc_rms = evaluate(rms_model, X_test, y_test)

acc_sgd, acc_adam, acc_rms

Convergence comparison

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history_sgd.history['val_loss'], label="SGD")
plt.plot(history_adam.history['val_loss'], label="Adam")
plt.plot(history_rms.history['val_loss'], label="RMSprop")

plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.legend()
plt.show()

Observations

Typical behaviour:

Adam converges fastest

RMSprop is close to Adam

SGD converges more slowly but more smoothly

2. Choosing the right optimizer – considerations

When selecting an optimizer, we must consider:

Convergence speed

Adam and RMSprop usually reach low loss faster

SGD may need more epochs

Stability

Adaptive optimizers handle varying gradient scales better

SGD may oscillate without momentum

Generalization

SGD with momentum often generalizes better

Adam may overfit or converge to sharper minima

Memory and complexity

Adam and RMSprop store extra moving averages

SGD uses minimal memory

Practical guideline
Scenario	Recommended optimizer
Fast prototyping	Adam
Very deep / noisy gradients	RMSprop or Adam
Large dataset, strong generalization focus	SGD + momentum
Fine-tuning	SGD
Final comparison summary
Optimizer	Convergence	Stability	Generalization	Tuning
SGD	Slow	Medium	High	Hard
RMSprop	Fast	High	Medium	Easy
Adam	Very fast	High	Medium	Very easy